In [ ]:
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get("OPENAI_API_KEY")
)


In [ ]:
target_language = "Spanish"          # change as needed
target_text = "Hello, how are you?"  # the text the user should have said

In [ ]:
# 1. Transcribe audio with Whisper
with open("sample.wav", "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
    )
transcription_text = transcription.text


In [ ]:
def ask_gpt(system_prompt, user_prompt):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content.strip()

In [ ]:
# 2. Translation
def translate_text(text, language):
    return ask_gpt(
        "You are a professional translator. Return only the translation.",
        f"Translate this into {language}:\n\n{text}",
    )

translated_text = translate_text(transcription_text, target_language)

In [ ]:
# 3. Grammar feedback
def get_grammar_feedback(text, language):
    return ask_gpt(
        f"You are a {language} grammar tutor. Find grammatical errors and "
        "return the corrected text, followed by a short explanation of changes.",
        text,
    )

grammar_feedback = get_grammar_feedback(translated_text, target_language)

In [ ]:
# 4. Pronunciation feedback
def get_pronunciation_feedback(transcribed, target):
    return ask_gpt(
        "You are a pronunciation coach. Compare what the learner said "
        "(from speech-to-text) with what they intended to say. Point out "
        "words that were likely mispronounced or missed, and give specific, "
        "encouraging tips for improvement.",
        f"Intended: {target}\nActually said: {transcribed}",
    )

pronunciation_feedback = get_pronunciation_feedback(transcription_text, target_text)

In [ ]:
print("Transcription:", transcription_text)
print("Translation:", translated_text)
print("Grammar:", grammar_feedback)
print("Pronunciation:", pronunciation_feedback)